# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset on second primary colorectal cancer in cancer survivors, utilizing the `mlcroissant` library.

### Dataset Source
The dataset schema (Croissant format) is sourced from:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their `@id` fields using the Croissant metadata. Each record set contains information such as tabular data, file objects, or documentation.

In [ ]:
# List all record sets (by @id) and their field @ids
record_sets = list(dataset.record_sets)
print("Available Record Sets (@id and name):")
for record_set in record_sets:
    print(f"  - {record_set['@id']}: {record_set.get('name', '')}")

print("\nFields for each Record Set:")
for record_set in record_sets:
    print(f"\nRecord Set @id: {record_set['@id']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # Ensure it's always a list
    for field in fields:
        if isinstance(field, dict):
            print(f"    Field @id: {field['@id']}  Name: {field.get('name','')}")
        else:
            print(f"    Field @id: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis (using the record set and field `@id`s). The main clinical dataset is typically the first or main tabular record set. Here, we'll extract all available record sets, storing each in a DataFrame.

In [ ]:
# Identify the record set(s) containing the clinical tabular data
# We'll extract from all record sets for demonstration

dataframes = {}
for record_set in record_sets:
    record_set_id = record_set["@id"]
    print(f"\nExtracting data from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load records: {e}")

# Display columns and preview of the main clinical dataset (assume the first record set is the main one)
main_record_set_id = list(dataframes.keys())[0] if dataframes else None
if main_record_set_id is not None and not dataframes[main_record_set_id].empty:
    print(f"\nColumns for main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some typical EDA tasks: filtering for high values in a numeric field (e.g., diagnostic interval, age), normalizing the field, and grouping by another categorical field (e.g., MSI status or sex).

*All columns and field references in the code below use their `@id`.*

In [ ]:
# Pick appropriate field @ids for analysis
# List columns to guide the selection
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print('Available columns:', df.columns.tolist())

    # For illustration, let's assume the numeric field is '@id': 'http://mlcommons.org/croissant/field/Age' (adjust as needed)
    # and a grouping field '@id': 'http://mlcommons.org/croissant/field/Sex' (adjust if the actual @ids differ)
    # Replace with actual @ids if different in this dataset

    numeric_field_id = None
    group_field_id = None

    # Attempt to detect the field for Age and Sex by column name (since @id likely included as column name)
    for col in df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
        elif 'sex' in col.lower() or 'gender' in col.lower():
            group_field_id = col

    print(f"Using numeric field: {numeric_field_id if numeric_field_id else '[NOT FOUND]'}")
    print(f"Using group field: {group_field_id if group_field_id else '[NOT FOUND]'}")

    # Proceed if both fields found
    if numeric_field_id and group_field_id:
        # Filter records where age > 60, for example
        threshold = 60
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id, group_field_id]].head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / \
                                                       filtered_df[numeric_field_id].astype(float).std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the grouping field and compute mean
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("Could not identify appropriate numeric or grouping fields.")

## 5. Visualization
Visualize the distribution of the selected numeric variable and compare distributions across groupings (e.g., age by sex).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id and group_field_id:
    plt.figure(figsize=(10,6))
    sns.histplot(data=df, x=numeric_field_id, hue=group_field_id, bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Discover and load record sets and fields by their `@id` from a Croissant schema,
- Extract tabular data and perform field-based filtering and normalization,
- Group and visualize data using standard Python data science libraries.

The FAIR^2 dataset empowers further clinicopathological research into second primary colorectal cancer in survivors. For deeper analyses, leverage the dataset's full schema with the `mlcroissant` API, always referencing data entities via their `@id` fields for reproducibility. Explore documentation and data-use restrictions before proceeding to advanced modeling or publishing.
